In [3]:
import numpy as np
from scipy.signal import find_peaks
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
import platform
import sys
import pandas as pd
import mne
from mne import Annotations

### Load signal annotations

In [4]:

def load_annotation_file(txt_path):  #1)
    """
    Charge un fichier d’annotations avec structure :
    <start_sec> <time_str> <stage_str> <code>
    Retourne les segments REM (début, fin) en secondes.
    """
    segments = []
    current_start = None

    with open(txt_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) >= 3:
            start_sec = float(parts[0])
            stage = parts[2].upper()

            if stage == 'REM':
                if current_start is None:
                    current_start = start_sec
            else:
                if current_start is not None:
                    segments.append((current_start, start_sec))
                    current_start = None

    # Fin du fichier
    if current_start is not None:
        segments.append((current_start, start_sec))

    return segments


def extract_rem_segments(txt_path): #2)
    return load_annotation_file(txt_path)

def load_signals_and_annotations(edf_path, annot_path): #3)
    raw = mne.io.read_raw_edf(edf_path, preload=True)
    rem_segments = extract_rem_segments(annot_path)
    print(f"Fichiers détectés : {edf_path} + {annot_path}")
    print(f"Durée fichier EDF : {raw.times[-1]:.2f} secondes")
    print(f"Premier segment REM à t={rem_segments[0][0]:.2f} secondes")

    return raw, rem_segments


### epochs REM de 4s

In [5]:
def segment_rem_in_windows(raw, rem_segments, window_sec=4, step_sec=2):
    """
    Découpe les segments REM fournis en fenêtres glissantes, avec debug.

    Args:
        raw (mne.io.Raw): objet Raw EEG complet
        rem_segments (list): liste de tuples (start_sec, end_sec) des périodes REM
        window_sec (float): durée d'une fenêtre (en secondes)
        step_sec (float): pas de décalage entre fenêtres (en secondes)

    Returns:
        list of mne.io.Raw: fenêtres extraites du signal
    """
    windows = []
    print(f"[INFO] Fenêtrage en fenêtres de {window_sec}s avec pas de {step_sec}s.")

    for seg_idx, (start, end) in enumerate(rem_segments):
        if end - start < window_sec:
            print(f"[WARNING] Segment REM #{seg_idx} trop court ({end - start:.2f}s), ignoré.")
            continue

        print(f"[INFO] Segment REM #{seg_idx}: de {start:.2f}s à {end:.2f}s")
        t = start
        while t + window_sec <= end:
            print(f"  >>> Fenêtre : tmin={t:.2f}s, tmax={t + window_sec:.2f}s")
            try:
                epoch = raw.copy().crop(tmin=t, tmax=t + window_sec).load_data()
                windows.append(epoch)
            except Exception as e:
                print(f"[ERROR] Impossible de couper la fenêtre [{t:.2f}s - {t + window_sec:.2f}s] : {e}")
            t += step_sec

    print(f"[INFO] Total de {len(windows)} fenêtres extraites.")
    return windows

### OEG criteria

In [6]:

def detect_eog_microstate(win, sfreq=250.0):
    """
    Classe une fenêtre REM de 4 s comme "phasic", "tonic", ou "ignore"
    selon la section 2.2 de l'article (Simor et al., 2021).

    Critères :
    - EOG filtré entre 0.3–10 Hz
    - "phasic" si ≥2 déflexions >150 µV et <500 ms dans chacune des 2 moitiés (2s)
    - "tonic" si aucune déflexion >25 µV dans les 2 moitiés
    - sinon "ignore"
    """
    try:
        eog_picks = win.copy().pick_types(eog=True)
        data = eog_picks.get_data()
        if data.shape[0] == 0:
            return "ignore"

        signal = data[0]  # un seul canal EOG
        n_samples = signal.shape[0]
        half = n_samples // 2

        def count_em(segment):
            peaks, _ = find_peaks(np.abs(segment), height=150)
            if len(peaks) < 2:
                return 0
            peak_times = peaks / sfreq
            durations = np.diff(peak_times)
            return np.sum(durations < 0.5)

        em_left = count_em(signal[:half])
        em_right = count_em(signal[half:])

        if em_left >= 2 and em_right >= 2:
            return "phasic"
        if np.max(np.abs(signal[:half])) < 25 and np.max(np.abs(signal[half:])) < 25:
            return "tonic"
        return "ignore"

    except Exception as e:
        print(f"[EOG ERROR] {e}")
        return "ignore"


### Annotations

In [8]:

def annotate_microstates(raw, windows, labels, window_sec):
    """
    Ajoute des annotations 'phasic' / 'tonic' à l'objet raw.

    Args:
        raw (mne.io.Raw): signal EEG original
        windows (list of mne.io.Raw): fenêtres découpées
        labels (list of str): liste de labels ('phasic' / 'tonic') pour chaque fenêtre
        window_sec (float): durée de chaque fenêtre
    """
    onset = [win.first_time for win in windows]
    duration = [window_sec] * len(windows)
    description = labels

    annotations = Annotations(onset=onset, duration=duration, description=description)
    raw.set_annotations(annotations)
    print(f"[INFO] {len(annotations)} annotations 'tonic/phasic' ajoutées à raw.")


In [ ]:

system = platform.system()
if system == "Darwin":
    disque = "/Volumes/Crucial X6"
elif system == "Windows":
    disque = "D:"
elif system == "Linux":
    disque = "/media/darryld/Crucial X6"
else:
    raise RuntimeError("Système non supporté.")

edf_path = Path(f"{disque}/EEG/raw/MN143/MN143_raw.edf")
annot_path = Path(f"{disque}/EEG/raw/MN143/MN143_hypnoEXP.txt")

print("Chargement des fichiers...")
raw, rem_segments = load_signals_and_annotations(edf_path, annot_path)

print(f"{len(rem_segments)} segments REM détectés.")
windows = segment_rem_in_windows(raw, rem_segments, window_sec=4, step_sec=4)

labels = []
valid_windows = []
for win in windows:
    label = detect_eog_microstate(win)
    if label != "ignore":
        labels.append(label)
        valid_windows.append(win)

print(f"{len(valid_windows)} fenêtres retenues ({labels.count('phasic')} phasic / {labels.count('tonic')} tonic)")

annotate_microstates(raw, valid_windows, labels, window_sec=4)

output_dir = Path(f"{disque}/EEG/preprocessed/{montage}/full")
output_dir.mkdir(parents=True, exist_ok=True)
annotated_path = output_dir / f"{edf_path.stem}_annotated.fif"
raw.save(annotated_path, overwrite=True)
print(f"[OK] Fichier annoté sauvegardé : {annotated_path}")

df = pd.DataFrame({
    "tmin": [win.first_time for win in valid_windows],
    "tmax": [win.first_time + 4 for win in valid_windows],
    "label": labels
})
df.to_excel(output_dir / f"{edf_path.stem}_microstates.xlsx", index=False)
print("[OK] Export Excel terminé.")



In [ ]:
def detect_eog_microstate(win, sfreq=250.0):
    """
    Classe une fenêtre REM de 4 s comme "phasic", "tonic", ou "ignore"
    selon la section 2.2 de l'article (Simor et al., 2021).

    Critères :
    - EOG filtré entre 0.3–10 Hz
    - "phasic" si ≥2 déflexions >150 µV et <500 ms dans chacune des 2 moitiés (2s)
    - "tonic" si aucune déflexion >25 µV dans les 2 moitiés
    - sinon "ignore"
    """
    try:
        eog_picks = win.copy().pick_types(eog=True)
        data = eog_picks.get_data()
        if data.shape[0] == 0:
            return "ignore"

        signal = data[0]  # un seul canal EOG
        n_samples = signal.shape[0]
        half = n_samples // 2

        def count_em(segment):
            peaks, _ = find_peaks(np.abs(segment), height=150)
            if len(peaks) < 2:
                return 0
            peak_times = peaks / sfreq
            durations = np.diff(peak_times)
            return np.sum(durations < 0.5)

        em_left = count_em(signal[:half])
        em_right = count_em(signal[half:])

        if em_left >= 2 and em_right >= 2:
            return "phasic"
        if np.max(np.abs(signal[:half])) < 25 and np.max(np.abs(signal[half:])) < 25:
            return "tonic"
        return "ignore"

    except Exception as e:
        print(f"[EOG ERROR] {e}")
        return "ignore"